# Robust Regression Engine

**Project:** Real-estate house-price prediction  
**Objective:** Build a robust regression pipeline covering regularization, cross-validation, tree-based regression, SVR, model comparison, and business interpretation.

> **Dataset note:** The assignment PDF references an external dataset link, but the uploaded PDF does not include a usable URL. To keep this project runnable end-to-end, this submission includes a clearly labeled synthetic real-estate dataset (`data/house_prices.csv`) with the same required type of features. Replace it with the official dataset if your instructor provides one, keeping `house_price` as the target column or updating `TARGET` below.


## Project roadmap

- Part A: Conceptual foundation
- Part B: Dataset understanding and preparation
- Part C: Ridge and Lasso regression with alpha tuning
- Part D: K-Fold, stratified-by-target-bin, LOOCV, and time-series split
- Part E: Decision Tree and Random Forest regression
- Part F: Linear-kernel and RBF-kernel SVR with hyperparameter tuning
- Part G: MSE, MAE, RMSE, R² and overfitting analysis
- Part H: Final conclusions and business interpretation


In [ ]:
# 1. Imports and configuration
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold, LeaveOneOut,
    TimeSeriesSplit, cross_val_score, GridSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

RANDOM_STATE = 42
TARGET = "house_price"
DATA_PATH = Path("data/house_prices.csv")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
# 2. Load dataset
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {DATA_PATH.resolve()}")

data = pd.read_csv(DATA_PATH)
display(data.head())
print("Shape:", data.shape)
display(data.describe().T)


## Part A — Conceptual foundation

### 1. Regularization
Regularization adds a penalty to large model coefficients. It reduces model complexity, controls overfitting, and often improves generalization to unseen data.

### 2. Ridge vs Lasso
Ridge uses an L2 penalty and usually shrinks coefficients toward zero without exactly removing many features. Lasso uses an L1 penalty and can make some coefficients exactly zero, supporting feature selection.

### 3. Cross-validation
Cross-validation repeatedly trains and validates a model on different data partitions. It gives a more stable estimate of generalization performance than relying on one validation split.

### 4. Cross-validation techniques
- **K-Fold:** Splits the data into K folds and uses each fold once as validation.
- **Stratified K-Fold:** Preserves class proportions; for regression, target values can first be binned into groups.
- **LOOCV:** Uses one observation as validation and all other observations for training in each iteration.
- **Time Series Split:** Uses earlier observations for training and later observations for validation, respecting temporal order.

### 5. Tree models and scaling
Decision trees split on feature thresholds, so they do not depend on distances or coefficient magnitudes. Therefore, monotonic scaling usually does not change their split logic, unlike linear models and SVR.


In [ ]:
# 3. Identify features and target
X = data.drop(columns=[TARGET])
y = data[TARGET]

print("Target:", TARGET)
print("Features:", list(X.columns))
print("Missing values:")
display(data.isna().sum().to_frame("missing_count"))


In [ ]:
# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ],
    remainder="drop"
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


## Part C — Regularized linear models

The alpha parameter controls the strength of regularization. Higher alpha means stronger coefficient shrinkage, which can reduce variance but may increase bias.


In [ ]:
# 5. Ridge and Lasso hyperparameter tuning
alpha_grid = {"model__alpha": np.logspace(-3, 4, 12)}

ridge_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Ridge())
])
lasso_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", Lasso(max_iter=100000))
])

ridge_search = GridSearchCV(
    ridge_pipe, alpha_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)
lasso_search = GridSearchCV(
    lasso_pipe, alpha_grid, cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1
)

ridge_search.fit(X_train, y_train)
lasso_search.fit(X_train, y_train)

print("Best Ridge:", ridge_search.best_params_)
print("Best Lasso:", lasso_search.best_params_)


In [ ]:
# 6. Metrics helper and linear model comparison
def regression_metrics(model_name, estimator, X_tr, y_tr, X_te, y_te):
    pred_train = estimator.predict(X_tr)
    pred_test = estimator.predict(X_te)
    return {
        "model": model_name,
        "train_mse": mean_squared_error(y_tr, pred_train),
        "test_mse": mean_squared_error(y_te, pred_test),
        "train_mae": mean_absolute_error(y_tr, pred_train),
        "test_mae": mean_absolute_error(y_te, pred_test),
        "train_rmse": mean_squared_error(y_tr, pred_train, squared=False),
        "test_rmse": mean_squared_error(y_te, pred_test, squared=False),
        "train_r2": r2_score(y_tr, pred_train),
        "test_r2": r2_score(y_te, pred_test),
    }

linear_results = pd.DataFrame([
    regression_metrics("Ridge", ridge_search.best_estimator_, X_train, y_train, X_test, y_test),
    regression_metrics("Lasso", lasso_search.best_estimator_, X_train, y_train, X_test, y_test),
])
display(linear_results)


### Coefficient behavior

- Ridge generally retains all features but reduces the magnitude of coefficients.
- Lasso may set some coefficients to zero.
- Because the preprocessing step scales numeric features, coefficient magnitudes are more comparable across numeric predictors.


In [ ]:
# 7. Inspect coefficients where available
def show_coefficients(fitted_pipeline, title):
    model = fitted_pipeline.named_steps["model"]
    prep = fitted_pipeline.named_steps["preprocessor"]
    try:
        names = prep.get_feature_names_out()
        coeffs = pd.Series(model.coef_, index=names).sort_values(key=np.abs, ascending=False)
        print(title)
        display(coeffs.head(15).to_frame("coefficient"))
    except Exception as exc:
        print("Coefficient inspection unavailable:", exc)

show_coefficients(ridge_search.best_estimator_, "Ridge coefficients")
show_coefficients(lasso_search.best_estimator_, "Lasso coefficients")


## Part D — Cross-validation strategies

For regression, stratification is approximated by binning the continuous target into quantile groups. Time-series validation is only conceptually appropriate when the data has a meaningful chronological ordering.


In [ ]:
# 8. Compare CV strategies using Ridge
X_all = X.reset_index(drop=True)
y_all = y.reset_index(drop=True)

ridge_model = ridge_search.best_estimator_

cv_results = []

# K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = -cross_val_score(ridge_model, X_all, y_all, cv=kf,
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_results.append({"strategy": "K-Fold", "mean_rmse": scores.mean(), "std_rmse": scores.std()})

# Stratified K-Fold using target bins
target_bins = pd.qcut(y_all, q=5, labels=False, duplicates="drop")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores = -cross_val_score(ridge_model, X_all, y_all, cv=skf.split(X_all, target_bins),
                          scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_results.append({"strategy": "Stratified K-Fold (target bins)", "mean_rmse": scores.mean(), "std_rmse": scores.std()})

# LOOCV on a reduced sample for practical runtime
small_n = min(150, len(X_all))
X_small, y_small = X_all.iloc[:small_n], y_all.iloc[:small_n]
loo = LeaveOneOut()
loo_scores = -cross_val_score(ridge_model, X_small, y_small, cv=loo,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_results.append({"strategy": "LOOCV (first 150 rows max)", "mean_rmse": loo_scores.mean(), "std_rmse": loo_scores.std()})

# Time Series Split after sorting by listing year
ordered = data.sort_values("listing_year").reset_index(drop=True)
X_time = ordered.drop(columns=[TARGET])
y_time = ordered[TARGET]
tss = TimeSeriesSplit(n_splits=5)
time_scores = -cross_val_score(ridge_model, X_time, y_time, cv=tss,
                               scoring="neg_root_mean_squared_error", n_jobs=-1)
cv_results.append({"strategy": "Time Series Split", "mean_rmse": time_scores.mean(), "std_rmse": time_scores.std()})

cv_comparison = pd.DataFrame(cv_results)
display(cv_comparison)


**Interpretation:** Lower mean RMSE is better. The standard deviation indicates how much validation performance varies across folds. A low mean RMSE with a lower standard deviation generally indicates more stable validation behavior, but the final choice should also consider the real deployment setting.


In [ ]:
# 9. Plot CV comparison
plt.figure(figsize=(10, 5))
plt.bar(cv_comparison["strategy"], cv_comparison["mean_rmse"], yerr=cv_comparison["std_rmse"], capsize=4)
plt.xticks(rotation=25, ha="right")
plt.ylabel("Mean RMSE")
plt.title("Cross-validation strategy comparison")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "cv_comparison.png", dpi=160)
plt.show()


## Part E — Tree-based regression models

Decision trees can model non-linear relationships but may overfit when allowed to grow without restrictions. Random Forest averages many trees, usually improving stability and reducing variance.


In [ ]:
# 10. Decision Tree and Random Forest tuning
tree_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeRegressor(random_state=RANDOM_STATE))
])
tree_grid = {
    "model__max_depth": [3, 5, 8, 12, None],
    "model__min_samples_leaf": [1, 3, 8, 15]
}
tree_search = GridSearchCV(tree_pipe, tree_grid, cv=5,
                           scoring="neg_root_mean_squared_error", n_jobs=-1)
tree_search.fit(X_train, y_train)

forest_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1))
])
forest_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 8, 15],
    "model__min_samples_leaf": [1, 3, 8]
}
forest_search = GridSearchCV(forest_pipe, forest_grid, cv=5,
                             scoring="neg_root_mean_squared_error", n_jobs=-1)
forest_search.fit(X_train, y_train)

print("Best tree:", tree_search.best_params_)
print("Best forest:", forest_search.best_params_)


In [ ]:
# 11. Tree comparison
tree_results = pd.DataFrame([
    regression_metrics("Decision Tree", tree_search.best_estimator_, X_train, y_train, X_test, y_test),
    regression_metrics("Random Forest", forest_search.best_estimator_, X_train, y_train, X_test, y_test),
])
display(tree_results)


## Part F — Support Vector Regression

SVR is sensitive to feature scale, so the numeric preprocessing pipeline includes standardization. `C` controls the penalty for errors, `epsilon` defines the epsilon-insensitive region, and `gamma` controls the influence range for RBF kernels.


In [ ]:
# 12. SVR: linear and RBF kernels
svr_linear = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVR(kernel="linear"))
])
svr_rbf = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SVR(kernel="rbf"))
])

linear_svr_grid = {
    "model__C": [0.1, 1, 10, 100],
    "model__epsilon": [0.01, 0.1, 0.5]
}
rbf_svr_grid = {
    "model__C": [1, 10, 100],
    "model__gamma": ["scale", 0.01, 0.1],
    "model__epsilon": [0.01, 0.1, 0.5]
}

svr_linear_search = GridSearchCV(svr_linear, linear_svr_grid, cv=3,
                                 scoring="neg_root_mean_squared_error", n_jobs=-1)
svr_rbf_search = GridSearchCV(svr_rbf, rbf_svr_grid, cv=3,
                              scoring="neg_root_mean_squared_error", n_jobs=-1)

svr_linear_search.fit(X_train, y_train)
svr_rbf_search.fit(X_train, y_train)

print("Best linear SVR:", svr_linear_search.best_params_)
print("Best RBF SVR:", svr_rbf_search.best_params_)


In [ ]:
# 13. SVR comparison
svr_results = pd.DataFrame([
    regression_metrics("SVR Linear", svr_linear_search.best_estimator_, X_train, y_train, X_test, y_test),
    regression_metrics("SVR RBF", svr_rbf_search.best_estimator_, X_train, y_train, X_test, y_test),
])
display(svr_results)


## Part G — Model comparison and evaluation

The metrics used are:
- **MSE:** Penalizes large errors strongly.
- **MAE:** Average absolute prediction error; easier to interpret in currency units.
- **RMSE:** Square root of MSE; expressed in the same unit as house price.
- **R²:** Proportion of target variance explained by the model.


In [ ]:
# 14. Final model comparison
all_estimators = [
    ("Ridge", ridge_search.best_estimator_),
    ("Lasso", lasso_search.best_estimator_),
    ("Decision Tree", tree_search.best_estimator_),
    ("Random Forest", forest_search.best_estimator_),
    ("SVR Linear", svr_linear_search.best_estimator_),
    ("SVR RBF", svr_rbf_search.best_estimator_),
]
all_results = pd.DataFrame([
    regression_metrics(name, est, X_train, y_train, X_test, y_test)
    for name, est in all_estimators
]).sort_values("test_rmse")
display(all_results)
all_results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)


In [ ]:
# 15. Generalization gap and overfitting analysis
analysis_table = all_results[[
    "model", "train_rmse", "test_rmse", "train_r2", "test_r2"
]].copy()
analysis_table["rmse_gap"] = analysis_table["test_rmse"] - analysis_table["train_rmse"]
analysis_table["r2_gap"] = analysis_table["train_r2"] - analysis_table["test_r2"]
display(analysis_table.sort_values("rmse_gap", ascending=False))


In [ ]:
# 16. Actual vs predicted plot for the selected test-best model
best_model_name = all_results.iloc[0]["model"]
best_estimator = dict(all_estimators)[best_model_name]
best_predictions = best_estimator.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.65)
lims = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual house price")
plt.ylabel("Predicted house price")
plt.title(f"Actual vs Predicted — {best_model_name}")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "actual_vs_predicted.png", dpi=160)
plt.show()


## Part H — Final analysis and business interpretation

1. **Best-performing model:** Select the model with the lowest test RMSE, while checking that the train-test gap is not excessively large.
2. **Impact of regularization:** Ridge and Lasso constrain coefficient size. This generally reduces variance and can make predictions more stable.
3. **Role of cross-validation:** Multiple folds expose performance variation across different samples and reduce dependence on one random split.
4. **Linear vs non-linear regressors:** Linear models are easier to interpret and work well when relationships are approximately additive. Trees and RBF-SVR can capture non-linear relationships but require complexity control and tuning.
5. **Business interpretation:** The model can support preliminary property pricing, portfolio screening, and valuation analysis. Predictions should be treated as estimates, not guaranteed market prices. Monitor drift, location changes, economic conditions, and data quality before production use.


In [ ]:
# 17. Save a concise final summary
best_row = all_results.iloc[0]
summary = pd.DataFrame([{
    "selected_model_by_test_rmse": best_row["model"],
    "test_rmse": best_row["test_rmse"],
    "test_mae": best_row["test_mae"],
    "test_r2": best_row["test_r2"],
    "note": "Selection is based on this synthetic dataset and should be revalidated with the official dataset."
}])
display(summary)
summary.to_csv(OUTPUT_DIR / "final_summary.csv", index=False)
